# Skill Benchmark Model

This notebook trains a regression model that estimates the expected number of technologies for an employed developer based on country, education, and coding experience.

In [2]:
import pandas as pd

df_raw = pd.read_csv("../../data/processed/stackoverflow_combined.csv")

In [6]:
df_clean = (
    df_raw
    .drop(columns=["Unnamed: 0"], errors="ignore")
    .drop_duplicates()
    .copy()
)

skill_features = [
    "Country",
    "EdLevel",
    "YearsCode",
    "YearsCodePro"
]

skill_target = "ComputerSkills"

skill_model_mask = (
    (df_clean["Employment"] == 1)
    & (df_clean["YearsCodePro"] <= df_clean["YearsCode"])
)

skill_df = df_clean.loc[
    skill_model_mask,
    skill_features + [skill_target]
].copy()

skill_df.shape

(39072, 5)

## Target Distribution

In [7]:
skill_target_summary = (
    skill_df["ComputerSkills"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .round(2)
)

skill_target_summary

count    39072.00
mean        17.26
std          6.67
min          3.00
1%           7.00
5%           9.00
25%         13.00
50%         16.00
75%         21.00
95%         29.00
99%         38.00
max        107.00
Name: ComputerSkills, dtype: float64

In [8]:
skill_upper_limit = skill_df["ComputerSkills"].quantile(0.99)

skill_training_df = skill_df.loc[
    skill_df["ComputerSkills"] <= skill_upper_limit
].copy()

skill_dataset_sizes = {
    "before_outlier_filter": len(skill_df),
    "after_outlier_filter": len(skill_training_df),
    "removed_rows": len(skill_df) - len(skill_training_df)
}

skill_dataset_sizes

{'before_outlier_filter': 39072,
 'after_outlier_filter': 38689,
 'removed_rows': 383}

In [9]:
from sklearn.model_selection import train_test_split

x = skill_training_df[skill_features]
y = skill_training_df[skill_target]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

x_train.shape, x_test.shape

((30951, 4), (7738, 4))

## Preprocessing and Baseline

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "Country",
    "EdLevel"
]

numerical_features = [
    "YearsCode",
    "YearsCodePro"
]

skill_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ],
    remainder="drop"
)

skill_preprocessor

ColumnTransformer(transformers=[('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Country', 'EdLevel']),
                                ('numerical', 'passthrough',
                                 ['YearsCode', 'YearsCodePro'])])

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

dummy_skill_pipeline = Pipeline(
    steps=[
        ("preprocessor", skill_preprocessor),
        ("model", DummyRegressor(strategy="median"))
    ]
)

dummy_skill_pipeline.fit(x_train, y_train)

dummy_skill_predictions = dummy_skill_pipeline.predict(x_test)

dummy_skill_mae = mean_absolute_error(
    y_test,
    dummy_skill_predictions
)

round(dummy_skill_mae, 2)

4.75

## Candidate Model — HistGradientBoostingRegressor

In [12]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

skill_model_pipeline = Pipeline(
    steps=[
        ("preprocessor", skill_preprocessor),
        (
            "model",
            HistGradientBoostingRegressor(
                loss="poisson",
                max_iter=200,
                learning_rate=0.08,
                max_leaf_nodes=31,
                min_samples_leaf=30,
                l2_regularization=1.0,
                random_state=42
            )
        )
    ]
)

skill_model_pipeline.fit(x_train, y_train)

skill_model_predictions = skill_model_pipeline.predict(x_test)

In [13]:
skill_model_comparison = pd.DataFrame(
    [
        {
            "model": "DummyRegressor",
            "mae": mean_absolute_error(y_test, dummy_skill_predictions),
            "rmse": root_mean_squared_error(y_test, dummy_skill_predictions),
            "r2": r2_score(y_test, dummy_skill_predictions)
        },
        {
            "model": "HistGradientBoostingRegressor",
            "mae": mean_absolute_error(y_test, skill_model_predictions),
            "rmse": root_mean_squared_error(y_test, skill_model_predictions),
            "r2": r2_score(y_test, skill_model_predictions)
        }
    ]
).set_index("model").round(3)

skill_model_comparison

,mae,rmse,r2
model,,,
DummyRegressor,4.747,6.192,-0.034
HistGradientBoostingRegressor,4.735,6.035,0.018


## Model Decision

The candidate model reduced MAE from 4.747 to only 4.735 and achieved an R² score of 0.018. Country, education, and experience therefore do not contain enough predictive signal to estimate an individual's exact skill count reliably.

Instead of deploying a weak predictive model, the product will use a descriptive cohort benchmark based on comparable employed developers.

In [14]:
skill_benchmark_df = skill_training_df.copy()

experience_bins = [-1, 1, 4, 9, 14, float("inf")]

experience_labels = [
    "0-1",
    "2-4",
    "5-9",
    "10-14",
    "15+"
]

skill_benchmark_df["ExperienceBand"] = pd.cut(
    skill_benchmark_df["YearsCodePro"],
    bins=experience_bins,
    labels=experience_labels
)

cohort_benchmark = (
    skill_benchmark_df
    .groupby(
        ["Country", "EdLevel", "ExperienceBand"],
        observed=True
    )["ComputerSkills"]
    .agg(
        cohort_size="count",
        expected_skills="median",
        lower_benchmark=lambda values: values.quantile(0.25),
        upper_benchmark=lambda values: values.quantile(0.75)
    )
    .reset_index()
)

reliable_cohort_benchmark = cohort_benchmark.loc[
    cohort_benchmark["cohort_size"] >= 30
].copy()

reliable_cohort_benchmark = reliable_cohort_benchmark.round(2)

reliable_cohort_benchmark.shape

(282, 7)

In [15]:
reliable_cohort_benchmark.loc[
    (reliable_cohort_benchmark["Country"] == "Turkey")
    & (reliable_cohort_benchmark["EdLevel"] == "Undergraduate")
]

,Country,EdLevel,ExperienceBand,cohort_size,expected_skills,lower_benchmark,upper_benchmark
1796,Turkey,Undergraduate,0-1,37,16.0,13.0,19.0
1797,Turkey,Undergraduate,2-4,129,17.0,13.0,22.0
1798,Turkey,Undergraduate,5-9,102,18.0,14.0,25.0
1799,Turkey,Undergraduate,10-14,67,18.0,13.0,23.0


## Benchmark Fallback Tables

The system first searches for a reliable country, education, and experience cohort. If that cohort contains fewer than 30 developers, it falls back to broader education-and-experience or experience-only benchmarks.

In [16]:
education_benchmark = (
    skill_benchmark_df
    .groupby(
        ["EdLevel", "ExperienceBand"],
        observed=True
    )["ComputerSkills"]
    .agg(
        cohort_size="count",
        expected_skills="median",
        lower_benchmark=lambda values: values.quantile(0.25),
        upper_benchmark=lambda values: values.quantile(0.75)
    )
    .reset_index()
)

education_benchmark = education_benchmark.loc[
    education_benchmark["cohort_size"] >= 30
].round(2)


experience_benchmark = (
    skill_benchmark_df
    .groupby(
        ["ExperienceBand"],
        observed=True
    )["ComputerSkills"]
    .agg(
        cohort_size="count",
        expected_skills="median",
        lower_benchmark=lambda values: values.quantile(0.25),
        upper_benchmark=lambda values: values.quantile(0.75)
    )
    .reset_index()
    .round(2)
)

benchmark_table_sizes = {
    "detailed_benchmarks": len(reliable_cohort_benchmark),
    "education_fallbacks": len(education_benchmark),
    "experience_fallbacks": len(experience_benchmark)
}

benchmark_table_sizes

{'detailed_benchmarks': 282,
 'education_fallbacks': 24,
 'experience_fallbacks': 5}

In [17]:
def get_experience_band(years_code_pro):
    if years_code_pro <= 1:
        return "0-1"
    elif years_code_pro <= 4:
        return "2-4"
    elif years_code_pro <= 9:
        return "5-9"
    elif years_code_pro <= 14:
        return "10-14"
    else:
        return "15+"

In [18]:
def get_skill_benchmark(country, education_level, years_code_pro):
    experience_band = get_experience_band(years_code_pro)

    detailed_match = reliable_cohort_benchmark.loc[
        (reliable_cohort_benchmark["Country"] == country)
        & (reliable_cohort_benchmark["EdLevel"] == education_level)
        & (reliable_cohort_benchmark["ExperienceBand"] == experience_band)
    ]

    if not detailed_match.empty:
        benchmark = detailed_match.iloc[0]
        benchmark_level = "country_education_experience"

    else:
        education_match = education_benchmark.loc[
            (education_benchmark["EdLevel"] == education_level)
            & (education_benchmark["ExperienceBand"] == experience_band)
        ]

        if not education_match.empty:
            benchmark = education_match.iloc[0]
            benchmark_level = "education_experience"

        else:
            experience_match = experience_benchmark.loc[
                experience_benchmark["ExperienceBand"] == experience_band
            ]

            benchmark = experience_match.iloc[0]
            benchmark_level = "experience"

    return {
        "benchmark_level": benchmark_level,
        "experience_band": experience_band,
        "cohort_size": int(benchmark["cohort_size"]),
        "expected_skills": int(round(benchmark["expected_skills"])),
        "lower_benchmark": int(round(benchmark["lower_benchmark"])),
        "upper_benchmark": int(round(benchmark["upper_benchmark"]))
    }

In [19]:
def analyze_skill_gap(
    country,
    education_level,
    years_code_pro,
    computer_skills
):
    result = get_skill_benchmark(
        country,
        education_level,
        years_code_pro
    )

    if computer_skills < result["lower_benchmark"]:
        position = "below_benchmark"
    elif computer_skills > result["upper_benchmark"]:
        position = "above_benchmark"
    else:
        position = "within_benchmark"

    result["actual_skills"] = computer_skills
    result["skill_gap"] = max(
        result["expected_skills"] - computer_skills,
        0
    )
    result["position"] = position

    return result

In [20]:
example_skill_analysis = analyze_skill_gap(
    country="Turkey",
    education_level="Undergraduate",
    years_code_pro=3,
    computer_skills=8
)

example_skill_analysis

{'benchmark_level': 'country_education_experience',
 'experience_band': '2-4',
 'cohort_size': 129,
 'expected_skills': 17,
 'lower_benchmark': 13,
 'upper_benchmark': 22,
 'actual_skills': 8,
 'skill_gap': 9,
 'position': 'below_benchmark'}

## Fallback Validation

In [23]:
detailed_test = analyze_skill_gap(
    country="Turkey",
    education_level="Undergraduate",
    years_code_pro=3,
    computer_skills=8
)

education_fallback_test = analyze_skill_gap(
    country="Turkey",
    education_level="Master",
    years_code_pro=1,
    computer_skills=8
)

experience_fallback_test = analyze_skill_gap(
    country="Turkey",
    education_level="UnknownLevel",
    years_code_pro=1,
    computer_skills=8
)

fallback_test_results = {
    "detailed":detailed_test["benchmark_level"],
    "education_fallback": education_fallback_test["benchmark_level"],
    "experience_fallback": experience_fallback_test["benchmark_level"]
}

fallback_test_results

{'detailed': 'country_education_experience',
 'education_fallback': 'education_experience',
 'experience_fallback': 'experience'}

In [24]:
assert detailed_test["benchmark_level"] == "country_education_experience"
assert education_fallback_test["benchmark_level"] == "education_experience"
assert experience_fallback_test["benchmark_level"] == "experience"

In [25]:
from pathlib import Path
import joblib

project_root = Path.cwd().parents[1]

skill_artifact_directory = (
    project_root / "artifacts" / "skills"
)

skill_artifact_directory.mkdir(
    parents=True,
    exist_ok=True
)

skill_benchmark_bundle = {
    "detailed_benchmark": reliable_cohort_benchmark,
    "education_benchmark": education_benchmark,
    "experience_benchmark": experience_benchmark,
    "experience_bins": experience_bins,
    "experience_labels": experience_labels,
    "minimum_cohort_size": 30,
    "training_rows": len(skill_benchmark_df),
    "outlier_upper_limit": float(skill_upper_limit),
    "version": "1.1.0"
}

skill_artifact_path = (
    skill_artifact_directory
    / "skill_benchmark_bundle_v2.joblib"
)

joblib.dump(
    skill_benchmark_bundle,
    skill_artifact_path
)

skill_artifact_path

PosixPath('/Users/ccakir/Desktop/codepath_ai/artifacts/skills/skill_benchmark_bundle_v1.joblib')

In [26]:
loaded_skill_bundle = joblib.load(skill_artifact_path)

loaded_skill_bundle.keys()

dict_keys(['detailed_benchmark', 'education_benchmark', 'experience_benchmark', 'experience_bins', 'experience_labels', 'minimum_cohort_size', 'training_rows', 'outlier_upper_limit', 'version'])